<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/04_Foundations/statistics/02_probability_distributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Probability Distributions

The language of uncertainty — master the distributions that appear everywhere in ML and statistics.

**Topics:** Discrete distributions (Bernoulli, Binomial, Poisson), continuous distributions (Normal, Exponential, Beta), CLT, distribution fitting

## 1. Discrete Distributions

In [ ]:
import numpy as np
from scipy import stats

# Bernoulli: single trial, binary outcome
p = 0.3  # P(success) = 0.3
bern = stats.bernoulli(p)
print('Bernoulli(p=0.3) — single coin flip')
print(f'  P(X=1) = {bern.pmf(1):.2f}, P(X=0) = {bern.pmf(0):.2f}')
print(f'  Mean = p = {bern.mean():.2f}, Var = p(1-p) = {bern.var():.2f}')
print()

# Binomial: n trials, each Bernoulli(p)
n, p = 20, 0.3
binom = stats.binom(n, p)
print(f'Binomial(n={n}, p={p}) — 20 email clicks, each p=0.3')
print(f'  P(X=6) = {binom.pmf(6):.4f}')
print(f'  P(X <= 5) = {binom.cdf(5):.4f}')
print(f'  Mean = np = {binom.mean():.1f}, Std = {binom.std():.2f}')
print()

# Poisson: count of rare events in fixed interval
lam = 4.5  # average 4.5 support tickets per hour
pois = stats.poisson(lam)
print(f'Poisson(λ={lam}) — support tickets per hour')
print(f'  P(X=5) = {pois.pmf(5):.4f}')
print(f'  P(X > 8) = {1 - pois.cdf(8):.4f}')
print(f'  Mean = λ = {pois.mean():.1f}, Var = λ = {pois.var():.1f} (mean = variance!)')
print()

# Real-world applications
applications = [
    ('Bernoulli', 'Click/no-click, spam/not-spam, defect/no-defect'),
    ('Binomial', 'A/B test conversions, quality control (N items, k defective)'),
    ('Poisson', 'Website requests/sec, bug count in code, call center arrivals'),
    ('Negative Binomial', 'Overdispersed counts (var > mean) — user sessions, word frequency'),
    ('Geometric', 'Trials until first success — churn after k emails'),
]
print('Real-world applications:')
for dist, uses in applications:
    print(f'  {dist:<22}: {uses}')

## 2. The Normal Distribution and Its Properties

In [ ]:
from scipy import stats
import numpy as np

mu, sigma = 100, 15  # IQ scores
norm = stats.norm(mu, sigma)

print('Normal(μ=100, σ=15) — IQ scores')
print()

# The 68-95-99.7 rule
print('68-95-99.7 rule:')
for n_sigma in [1, 2, 3]:
    prob = norm.cdf(mu + n_sigma * sigma) - norm.cdf(mu - n_sigma * sigma)
    print(f'  μ ± {n_sigma}σ ({mu-n_sigma*sigma}–{mu+n_sigma*sigma}): {prob*100:.1f}% of data')
print()

# Z-score standardization
def z_score(x, mu, sigma): return (x - mu) / sigma

iq_value = 130
z = z_score(iq_value, mu, sigma)
percentile = norm.cdf(iq_value) * 100
print(f'IQ = {iq_value}:')
print(f'  Z-score = {z:.2f} standard deviations above mean')
print(f'  Percentile = {percentile:.1f}th (better than {percentile:.1f}% of population)')
print(f'  P(X > {iq_value}) = {(1 - norm.cdf(iq_value))*100:.1f}%')
print()

# Log-normal: when to use it instead
print('Normal vs Log-Normal:')
cases = [
    ('Normal', ['Heights', 'IQ scores', 'Measurement errors', 'Returns over short periods']),
    ('Log-Normal', ['Income', 'House prices', 'Stock prices', 'Website session durations', 'Bug counts']),
]
for dist, examples in cases:
    print(f'  {dist}: {", ".join(examples)}')
print()
print('Rule: if X can only be positive and has a long right tail → try log(X) first')

## 3. Central Limit Theorem — Why the Normal Distribution is Everywhere

In [ ]:
import numpy as np
from scipy import stats

def clt_demonstration(population_dist, dist_name: str, sample_sizes: list[int], n_simulations: int = 5000):
    """Show CLT: sample means become normal regardless of population distribution."""
    print(f'Population: {dist_name}')
    pop = population_dist(10_000)
    print(f'  Skewness: {stats.skew(pop):.2f}, Kurtosis: {stats.kurtosis(pop):.2f}')
    print()
    print(f'  {"Sample size n":<16} {"Mean of means":<18} {"Std of means":<16} {"Normal test p-val"}')
    print('  ' + '-' * 70)
    
    for n in sample_sizes:
        sample_means = [population_dist(n).mean() for _ in range(n_simulations)]
        _, p_norm = stats.normaltest(sample_means)
        print(f'  n={n:<14} {np.mean(sample_means):<18.3f} {np.std(sample_means):<16.4f} {p_norm:.4f} {"← Normal!" if p_norm > 0.05 else "← not yet"}')

# Exponential distribution (very skewed) → means become normal
clt_demonstration(
    lambda n: np.random.exponential(scale=2, size=n),
    'Exponential(λ=0.5) — very right skewed',
    [1, 5, 10, 30, 100],
)
print()
print('CLT in practice:')
print('  ✓ Enables t-tests, confidence intervals — even on non-normal data')
print('  ✓ Explains why many sample statistics (mean, proportion) are normally distributed')
print('  ✓ Rule of thumb: n >= 30 is usually sufficient for CLT to kick in')
print('  ✗ Does NOT apply to extreme outliers or infinite-variance distributions')

## 4. Other Key Distributions in ML

In [ ]:
from scipy import stats
import numpy as np

# t-distribution: heavier tails than Normal, used with small samples
df_values = [1, 5, 10, 30, np.inf]  # degrees of freedom
print('t-distribution vs Normal (tails get lighter as df → ∞):')
print('P(|t| > 2) for various df:')
for df in df_values:
    if df == np.inf:
        p = 2 * (1 - stats.norm.cdf(2))
        name = 'Normal'
    else:
        p = 2 * (1 - stats.t.cdf(2, df=df))
        name = f't(df={df})'
    print(f'  {name:<15}: {p:.4f}')
print()

# Beta distribution: models probabilities, proportions
print('Beta(α, β) — models probabilities:')
beta_configs = [
    (1, 1, 'Uniform (no prior information)'),
    (2, 8, 'Low conversion rate prior (~20%)'),
    (50, 50, 'Confident 50/50 prior (many observations)'),
    (0.5, 0.5, 'Bimodal — U-shaped (Jeffreys prior)'),
]
for alpha, beta, desc in beta_configs:
    dist = stats.beta(alpha, beta)
    print(f'  Beta({alpha}, {beta}): mean={dist.mean():.2f}, std={dist.std():.2f}  # {desc}')
print()

# Chi-squared: sum of squared normals, used in hypothesis tests
# F-distribution: ratio of chi-squareds, used in ANOVA
print('Distribution genealogy:')
print('  Z ~ Normal(0,1)')
print('  Z² ~ Chi-squared(df=1)')
print('  Σ Zi² ~ Chi-squared(df=n)  → goodness-of-fit tests')
print('  Chi2(d1)/d1 / Chi2(d2)/d2 ~ F(d1,d2)  → ANOVA, regression F-test')
print('  Normal/sqrt(Chi2/df) ~ t(df)  → t-tests')

## 5. Distribution Fitting and Model Selection

In [ ]:
from scipy import stats
import numpy as np

def fit_and_compare(data: np.ndarray, distributions: list[str]) -> list[tuple]:
    """Fit multiple distributions and rank by AIC."""
    results = []
    for dist_name in distributions:
        dist = getattr(stats, dist_name)
        try:
            params = dist.fit(data)
            log_lik = dist.logpdf(data, *params).sum()
            k = len(params)  # number of parameters
            aic = 2 * k - 2 * log_lik
            ks_stat, ks_p = stats.kstest(data, dist_name, args=params)
            results.append((dist_name, params, aic, ks_stat, ks_p))
        except Exception:
            pass
    return sorted(results, key=lambda x: x[2])  # sort by AIC (lower = better)

# Generate log-normal data (simulating user session durations in seconds)
np.random.seed(42)
session_durations = stats.lognorm.rvs(s=1.2, loc=0, scale=60, size=1000)

candidate_distributions = ['norm', 'lognorm', 'expon', 'gamma', 'weibull_min']
fit_results = fit_and_compare(session_durations, candidate_distributions)

print('Distribution fitting — user session durations (seconds):')
print(f'{"Distribution":<15} {"AIC":>10} {"KS statistic":>14} {"KS p-value":>12} {"Verdict"}')
print('-' * 65)
for name, params, aic, ks_stat, ks_p in fit_results:
    verdict = '✓ Good fit' if ks_p > 0.05 else '✗ Poor fit'
    print(f'{name:<15} {aic:>10.1f} {ks_stat:>14.4f} {ks_p:>12.4f}  {verdict}')

print()
print('Model selection criteria:')
print('  AIC (lower = better): balances fit quality vs model complexity')
print('  KS test p > 0.05: cannot reject that data follows the distribution')
print('  Visual QQ-plot: best way to see where the fit fails')

## Exercises

1. **A/B test simulation:** You run an A/B test with control CTR = 5% and treatment CTR = 5.5%. Use the Binomial distribution to simulate 10,000 experiments with n=1000 users each. What fraction of experiments correctly detect the improvement? This is statistical power.

2. **Poisson process:** Model website traffic as a Poisson process with λ=100 requests/minute. Simulate 60 minutes of traffic. Compute the probability that any single minute exceeds 120 requests. What arrival rate (λ) would you need to handle if you want 99.9% uptime with capacity 150 req/min?

3. **Distribution fitting pipeline:** Download a real dataset (e.g., NYC taxi ride durations). Fit 5 candidate distributions using MLE. Use QQ-plots and AIC to select the best. Explain what the fitted distribution parameters mean in business terms.